In [12]:
!pip install -q transformers torchvision albumentations

In [13]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [14]:
!git clone https://github.com/tangsanli5201/DeepPCB.git

fatal: destination path 'DeepPCB' already exists and is not an empty directory.


In [15]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset


class DeepPCBDataset(Dataset):
    def __init__(self, root_dir, split_file):
        self.root_dir = root_dir

        with open(split_file, "r") as f:
            self.samples = [line.strip().split() for line in f.readlines()]

        # 6 clases
        self.classes = ["open", "short", "mousebite", "spur", "copper", "pin-hole"]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_rel_path, ann_rel_path = self.samples[idx]

        img_path = os.path.join(
            self.root_dir,
            img_rel_path.replace(".jpg", "_test.jpg")
        )
        ann_path = os.path.join(self.root_dir, ann_rel_path)

        image = Image.open(img_path).convert("RGB")

        annotations = []

        if os.path.exists(ann_path):
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split()

                    if len(parts) < 5:
                        continue

                    xmin, ymin, xmax, ymax, class_id = parts

                    xmin, ymin, xmax, ymax = map(float, [xmin, ymin, xmax, ymax])
                    class_id = int(class_id)

                    width = xmax - xmin
                    height = ymax - ymin

                    # evitar cajas inválidas
                    if width < 1 or height < 1:
                        continue

                    annotations.append({
                        "bbox": [xmin, ymin, width, height],
                        "category_id": class_id,
                        "area": width * height,
                        "iscrowd": 0
                    })

        target = {
            "image_id": idx,
            "annotations": annotations
        }

        return image, target

In [16]:
ROOT_DIR = "/content/DeepPCB/PCBData"
SPLIT_FILE = "/content/DeepPCB/PCBData/trainval.txt"

dataset = DeepPCBDataset(ROOT_DIR, SPLIT_FILE)

print("Samples:", len(dataset))
print("Clases:", dataset.classes)

Samples: 1000
Clases: ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']


In [17]:
img, target = dataset[0]

print(img.size)
print(target)

(640, 640)
{'image_id': 0, 'annotations': [{'bbox': [409.0, 394.0, 26.0, 28.0], 'category_id': 3, 'area': 728.0, 'iscrowd': 0}, {'bbox': [275.0, 383.0, 44.0, 34.0], 'category_id': 3, 'area': 1496.0, 'iscrowd': 0}, {'bbox': [8.0, 163.0, 28.0, 28.0], 'category_id': 4, 'area': 784.0, 'iscrowd': 0}, {'bbox': [244.0, 151.0, 26.0, 31.0], 'category_id': 5, 'area': 806.0, 'iscrowd': 0}, {'bbox': [338.0, 519.0, 26.0, 24.0], 'category_id': 6, 'area': 624.0, 'iscrowd': 0}, {'bbox': [476.0, 460.0, 26.0, 21.0], 'category_id': 4, 'area': 546.0, 'iscrowd': 0}]}


In [18]:
import torch
from torch.utils.data import random_split

# Fijar semilla para reproducibilidad
torch.manual_seed(42)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 800
Test size: 200


In [19]:
import torch
from transformers import DetrForObjectDetection, DetrImageProcessor

# Processor
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

# Mapeo de clases (6 defectos)
id2label = {
    0: "open",
    1: "short",
    2: "mousebite",
    3: "spur",
    4: "copper",
    5: "pin-hole",
}

label2id = {v: k for k, v in id2label.items()}

# Modelo con 6 clases
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Modelo listo en:", device)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                                         | Status     |                                                                                      
----------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------
model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer3.0.downsam

Modelo listo en: cuda


In [20]:
from torch.utils.data import DataLoader

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

def collate_fn(batch):
    images = [item[0] for item in batch]
    annotations = [item[1] for item in batch]

    encoding = processor(
        images=images,
        annotations=annotations,
        return_tensors="pt"
    )

    return {
        "pixel_values": encoding["pixel_values"],
        "labels": encoding["labels"]
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print("Collate y loaders listos")

Collate y loaders listos


In [ ]:
import torch

import json

import os

from torch.utils.data import DataLoader, random_split

from torchmetrics.detection.mean_ap import MeanAveragePrecision

from torchvision.ops import box_iou

from google.colab import drive



# =========================

# DRIVE

# =========================

drive.mount('/content/drive')



# =========================

# CONFIG

# =========================

lr = 5e-5

batch_size = 4

epochs = 30

patience = 3



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)



optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)



# =========================

# SAVE DIR

# =========================

save_dir = "/content/drive/MyDrive/entrenamiento_detr"

os.makedirs(save_dir, exist_ok=True)



# =========================

# SPLIT DATASET

# =========================

train_size = int(0.8 * len(dataset))

val_size = len(dataset) - train_size



train_dataset, val_dataset = random_split(

    dataset,

    [train_size, val_size],

    generator=torch.Generator().manual_seed(42)

)



# =========================

# DATALOADERS

# =========================

train_loader = DataLoader(

    train_dataset,

    batch_size=batch_size,

    shuffle=True,

    collate_fn=collate_fn,

    num_workers=0,

    pin_memory=True

)



val_loader = DataLoader(

    val_dataset,

    batch_size=batch_size,

    shuffle=False,

    collate_fn=collate_fn,

    num_workers=0,

    pin_memory=True

)



print("Train batches:", len(train_loader))

print("Val batches:", len(val_loader))



# =========================

# HISTORIAL

# =========================

history = {

    "train_loss": [],

    "val_loss": [],

    "map": [],

    "map_50": [],

    "map_75": [],

    "iou": []

}



config = {

    "learning_rate": lr,

    "batch_size": batch_size,

    "epochs": epochs,

    "optimizer": "AdamW"

}



# =========================

# BEST MODEL + EARLY STOP

# =========================

best_val_loss = float("inf")

epochs_no_improve = 0



# =========================

# TRAIN LOOP

# =========================

for epoch in range(epochs):



    # -------- TRAIN --------

    model.train()

    total_train_loss = 0.0



    for batch in train_loader:

        pixel_values = batch["pixel_values"].to(device)



        labels = [

            {k: v.to(device) for k, v in t.items()}

            for t in batch["labels"]

        ]



        outputs = model(pixel_values=pixel_values, labels=labels)

        loss = outputs.loss



        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()



        total_train_loss += loss.item()



    avg_train_loss = total_train_loss / max(len(train_loader), 1)



    # -------- VALIDATION + MÉTRICAS --------

    model.eval()

    total_val_loss = 0.0



    metric = MeanAveragePrecision(

        box_format="xyxy",

        iou_type="bbox"

    )

    metric.reset()



    total_iou = 0.0

    total_boxes = 0



    with torch.no_grad():

        for batch in val_loader:

            pixel_values = batch["pixel_values"].to(device)



            labels = [

                {k: v.to(device) for k, v in t.items()}

                for t in batch["labels"]

            ]



            # LOSS

            outputs = model(pixel_values=pixel_values, labels=labels)

            loss = outputs.loss

            total_val_loss += loss.item()



            # usar mismo output (sin doble forward)

            outputs_pred = outputs



            # tamaños reales CORRECTOS

            target_sizes = torch.stack([

                t["orig_size"] for t in batch["labels"]

            ]).to(device)



            results = processor.post_process_object_detection(

                outputs_pred,
                threshold=0.01,
                target_sizes=target_sizes

            )



            preds = []

            targets = []



            for i in range(len(results)):

                preds.append({

                    "boxes": results[i]["boxes"].cpu(),

                    "scores": results[i]["scores"].cpu(),

                    "labels": results[i]["labels"].cpu()

                })



                gt_boxes = batch["labels"][i]["boxes"].clone()

                h, w = batch["labels"][i]["orig_size"]

                # si están normalizados (0–1)
                if gt_boxes.max() <= 1.0:
                    # cxcywh → xyxy
                    cx, cy, bw, bh = gt_boxes.unbind(1)
                    x1 = cx - 0.5 * bw
                    y1 = cy - 0.5 * bh
                    x2 = cx + 0.5 * bw
                    y2 = cy + 0.5 * bh
                    gt_boxes = torch.stack([x1, y1, x2, y2], dim=1)

                    # pasar a píxeles
                    gt_boxes[:, [0, 2]] *= w
                    gt_boxes[:, [1, 3]] *= h

                targets.append({
                    "boxes": gt_boxes.cpu(),
                    "labels": batch["labels"][i]["class_labels"].cpu()
                })



                # ===== IoU =====

                pred_boxes = results[i]["boxes"].cpu()

                gt_boxes = batch["labels"][i]["boxes"].cpu()



                if len(pred_boxes) > 0 and len(gt_boxes) > 0:

                    ious = box_iou(pred_boxes, gt_boxes)

                    max_ious, _ = ious.max(dim=1)



                    total_iou += max_ious.sum().item()

                    total_boxes += len(max_ious)



            if any(len(p["boxes"]) > 0 for p in preds):
                metric.update(preds, targets)



    avg_val_loss = total_val_loss / max(len(val_loader), 1)

    avg_iou = total_iou / max(total_boxes, 1)



    metrics_result = metric.compute()

    map_score = metrics_result["map"].item()

    map_50 = metrics_result["map_50"].item()

    map_75 = metrics_result["map_75"].item()



    # -------- HISTORIAL --------

    history["train_loss"].append(avg_train_loss)

    history["val_loss"].append(avg_val_loss)

    history["map"].append(map_score)

    history["map_50"].append(map_50)

    history["map_75"].append(map_75)

    history["iou"].append(avg_iou)



    print(f"\nEpoch {epoch+1}/{epochs}")

    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    print(f"mAP: {map_score:.4f} | mAP@50: {map_50:.4f} | mAP@75: {map_75:.4f} | IoU: {avg_iou:.4f}")



    # -------- GUARDAR MEJOR MODELO --------

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        epochs_no_improve = 0



        torch.save({

            "model_state_dict": model.state_dict(),

            "epoch": epoch,

            "val_loss": avg_val_loss,

            "map": map_score,

            "iou": avg_iou

        }, os.path.join(save_dir, "best_model.pth"))



        print("Mejor modelo guardado")



    else:

        epochs_no_improve += 1



    # -------- EARLY STOPPING --------

    if epochs_no_improve >= patience:

        print("Early stopping activado")

        break



# =========================

# GUARDADO FINAL

# =========================

torch.save(model.state_dict(), os.path.join(save_dir, "model_final.pth"))



with open(os.path.join(save_dir, "history.json"), "w") as f:

    json.dump(history, f)



with open(os.path.join(save_dir, "config.json"), "w") as f:

    json.dump(config, f)



print("Todo guardado en Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train batches: 200
Val batches: 50

Epoch 1/30
Train Loss: 2.2314 | Val Loss: 1.7821
mAP: 0.0162 | mAP@50: 0.0304 | mAP@75: 0.0177 | IoU: 0.0000
Mejor modelo guardado

Epoch 2/30
Train Loss: 1.8207 | Val Loss: 1.7010
mAP: 0.0218 | mAP@50: 0.0407 | mAP@75: 0.0198 | IoU: 0.0000
Mejor modelo guardado

Epoch 3/30
Train Loss: 1.6733 | Val Loss: 1.6120
mAP: 0.0418 | mAP@50: 0.0704 | mAP@75: 0.0449 | IoU: 0.0000
Mejor modelo guardado


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 8))

# ---- LOSS ----
plt.subplot(2, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# ---- mAP ----
plt.subplot(2, 2, 2)
plt.plot(epochs, history["map"], label="mAP")
plt.plot(epochs, history["map_50"], label="mAP@50")
plt.title("mAP")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()

# ---- IoU ----
plt.subplot(2, 2, 3)
plt.plot(epochs, history["iou"], label="IoU")
plt.title("IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

# Tomar una muestra
image, target = dataset[10]

# DETR espera lista de imágenes
inputs = processor(images=[image], return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-processing
results = processor.post_process_object_detection(
    outputs,
    target_sizes=[(image.size[1], image.size[0])],  # (h, w)
    threshold=0.5
)[0]

# Plot
plt.figure(figsize=(8, 8))
plt.imshow(image)

# Si no hay detecciones, evitar crash
if len(results["boxes"]) == 0:
    print("No se detectaron objetos")

for score, label, box in zip(
    results["scores"],
    results["labels"],
    results["boxes"]
):
    xmin, ymin, xmax, ymax = box.cpu().numpy()

    # Dibujar bounding box
    plt.gca().add_patch(
        plt.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            edgecolor="red",
            linewidth=2
        )
    )

    # Mostrar label + score
    class_name = model.config.id2label[label.item()]
    score_value = score.item()

    plt.text(
        xmin,
        ymin,
        f"{class_name}: {score_value:.2f}",
        color="red",
        fontsize=9,
        bbox=dict(facecolor="yellow", alpha=0.5)
    )

plt.axis("off")
plt.title("Predicciones DETR")
plt.show()